# Week 8: Modern Deep RL Exploration

Craig Rudman  
Regis University  
MSDS684 Reinforcement Learning  
Prof. Mike Busch  

## Section 1: Project Overview

This course has traced a progression in reinforcement learning from tabular methods to function approximation. First with tile coding, then with one-step actor-critic, we moved from exact value representations to learned approximations capable of generalizing across continuous state spaces. Deep Q-Networks (DQN) mark another step change in that progression, adding experience replay to the established combination of TD bootstrapping and off-policy greedy control. This report compares DQN against a one-step actor-critic baseline on LunarLander-v3, using 20 seeds and 100,000 timesteps for both. Through hyperparameter sweeps across learning rate, exploration fraction, and target update interval, and ablation studies disabling experience replay and the target network, I demonstrate that DQN's architecture produces more stable and reliable learning than the policy gradient approach developed in Week 6. It continues to improve where actor-critic plateaus. The comparison also offers a decision framework: DQN's off-policy replay makes it more sample-efficient for discrete action spaces, while actor-critic methods generalize naturally to continuous control. LunarLander-v3's discrete action space favors DQN, and the results confirm it.

Of DQN's two key innovations, experience replay and the target network, experience replay does the heavier lifting. As Mnih et al. (2013) observe, “learning directly from consecutive samples is inefficient, due to the strong correlations between the samples; randomizing the samples breaks these correlations and therefore reduces the variance of the updates.” In other words, without experience replay, each training update reflects a narrow slice of the state space rather than its full diversity. This violates the assumption that gradient descent samples should be independent, producing oscillating, unstable weight updates. By storing transitions in a replay buffer and sampling randomly, DQN breaks the correlation between successive training updates and allows successful late-episode transitions to be replayed repeatedly, propagating credit backward to the early decisions that caused them.

The target network plays a complementary but secondary role. Sutton and Barto warn that “the danger of instability and divergence arises whenever we combine all of the following three elements... function approximation, bootstrapping, and off-policy training” (Sutton and Barto, p. 264). DQN sits squarely in all three legs of this deadly triad: the neural network provides function approximation, TD updates provide bootstrapping, and the replay buffer makes training off-policy by definition. Critically, Sutton and Barto note that “if any two elements of the deadly triad are present, but not all three, then instability can be avoided” (Sutton and Barto, p. 264). The target network partially mitigates the bootstrapping leg by freezing TD targets for 250 steps at a time, but it does not eliminate it. All three legs remain. The ablation confirms the asymmetry: without the target network, learning degrades but does not collapse. Without experience replay, it collapses.

Hyperparameter sensitivity reveals a subtler finding. The rl-baselines3-zoo published baseline for LunarLander reports a mean return of 136.79 from a single training run. A 20-seed experiment using the same configuration produces a mean of 105.0 with a standard deviation of 139.9, a coefficient of variation exceeding 100%. The zoo's single-run number is not wrong, but it is not representative. As shown in Section 3, the true performance distribution is wide, right-skewed, and highly seed-dependent. This is a structural property of DQN, not random noise. Understanding where that variance comes from, and how hyperparameter choices affect it, is the central story of these results.

The three hyperparameter sweeps reveal a consistent pattern that can be understood through the lens of bias and variance. Configurations that fit more aggressively to recent experience produced higher variance across seeds, while more conservative configurations generalized more reliably. Also shown in Section 3, this pattern holds across learning rate, exploration fraction, and target update interval, and it has a practical implication: the published zoo baseline, despite being tuned for this environment, appears optimized for a single run rather than reliable convergence. Accounting for DQN's structural variance, not just expected return, is one of the central lessons of this study.

The main takeaway of this study is methodological as much as algorithmic. Modern deep RL libraries make it straightforward to train an agent that sometimes solves a problem. Making it solve the problem reliably requires understanding the theoretical sources of variance, running enough seeds to characterize the full performance distribution, and recognizing that a published benchmark number is a point estimate from a potentially very wide distribution. DQN's innovations (experience replay, target networks, neural function approximation) are engineering solutions to the instabilities Sutton and Barto predict. They work, but they work probabilistically, and the gap between a lucky run and a reliable algorithm is larger than any single learning curve can show. Sutton and Barto's theory predicts instability; DQN's engineering solutions suppress it but they do not eliminate it. The divergence between textbook prediction and practical outcome is not that DQN contradicts the theory, but that it mitigates instability rather than resolving it. A practitioner reading only the theory would expect failure. A practitioner reading only a benchmark number would expect reliability. Neither picture is complete.

## Section 2: Code Description

### GitHub Repository

GitHub Repository: https://github.com/craig-rudman/MSDS684/tree/main/W8

### Implementation Summary

This study uses Stable-Baselines3's DQN implementation, baselined on the rl-baselines3-zoo configuration for LunarLander-v3. The zoo config provided a published starting point with known performance, making it a natural anchor for systematic experimentation. Rather than tuning from scratch, I adopted the zoo's hyperparameters directly and varied them as logarithmic steps above and below the baseline value. This approach ensures that each sweep covers a meaningful range relative to the baseline while remaining interpretable: a factor-of-three change in learning rate or exploration fraction is a qualitatively different regime, not an incremental adjustment.

The neural network architecture follows the zoo config: two hidden layers of 256 units each with ReLU activations. This may be larger than strictly necessary for LunarLander, but provides enough capacity to represent the Q-function across the full eight-dimensional state space without becoming a bottleneck. The zoo config also specifies gradient_steps=-1, which performs one gradient update per environment step collected, maintaining a one-to-one ratio of experience collection to learning updates throughout training. The replay buffer holds 50,000 transitions, and the target network updates every 250 steps.

Three hyperparameters were swept: learning rate (2e-4, 6.3e-4, 2e-3), exploration fraction (0.06, 0.12, 0.24), and target update interval (125, 250, 500). Each sweep varied one parameter while holding the others at zoo baseline values. Two ablation studies removed experience replay and the target network respectively, replacing each with the minimal viable alternative: a buffer of size one for the replay ablation, and a target update interval of 100,000 steps (effectively frozen for the full training run) for the target ablation.

SB3's DQN uses a constant learning rate throughout training with no annealing schedule, Adam as the optimizer, and no observation normalization for continuous state inputs like LunarLander's eight-dimensional vector. Reward scaling is also absent by default. These are deliberate omissions rather than oversights: the replay buffer's decorrelating effect is sufficient to manage the instability that normalization would otherwise address.

The primary challenge was computational. Running 20 seeds at 100,000 timesteps per seed across the baseline, three sweeps, and two ablations required approximately 24 hours of continuous training on an M3 Pro GPU. This made iteration expensive: a misconfigured experiment could not be caught quickly. I addressed this with a smoke test guard that runs two seeds for 2,000 steps before committing to a full run, catching configuration errors in seconds rather than hours.

## Section 3: Results

### 3.1 Baseline: DQN vs. Actor-Critic

The baseline experiment trained DQN using the rl-baselines3-zoo configuration across 20 random seeds for 100,000 timesteps on LunarLander-v3. The same protocol was applied to the one-step actor-critic implementation from Week 6, run on LunarLander-Continuous-v3, to provide a direct comparison of the two approaches under identical seed and budget conditions.

![Figure 1: Baseline comparison: DQN vs. Actor-Critic](../img/baseline_learning_curves.png)

*Figure 1. Smoothed mean episode return with 95% confidence interval across 20 seeds. Baseline DQN uses the zoo config (lr=6.3e-4, ef=0.12, tui=250, buffer=50k) on LunarLander-v3 (discrete). One-step actor-critic (actor_lr=1e-4, critic_lr=1e-3, gamma=0.99) is shown on LunarLander-Continuous-v3. Dashed line marks the solved threshold (+200).*

DQN improves steadily across the full 100,000 timesteps, reaching a mean return of 105.0 (std=139.9) by the end of training, with no sign of convergence. The confidence interval fans out sharply around 25,000 steps, the point at which epsilon decay ends and seed-specific Q-values begin to drive behavior. Actor-critic, by contrast, plateaus near -84 by 60,000 steps and makes no further progress across the remaining 40,000 timesteps. Its confidence interval is tight, reflecting low variance but also a low ceiling. DQN's mean return exceeds actor-critic's by roughly 190 points, and its trajectory is still rising where actor-critic has stalled entirely.

### 3.2 Learning Rate Sweep

The learning rate was varied across three values spanning two orders of magnitude: 2e-4 (one-third of baseline), 6.3e-4 (zoo baseline), and 2e-3 (approximately three times baseline). All other hyperparameters were held at zoo baseline values.

![Figure 2: Learning rate sweep](../img/sweep_lr_learning_curves.png)

*Figure 2. Smoothed mean episode return with 95% confidence interval across 20 seeds for each learning rate condition. All other hyperparameters held at zoo baseline values.*

The high learning rate (lr=2e-3) diverges immediately after epsilon decay ends, collapsing to a mean return of -241 by 100k steps. The two lower learning rates tell a more nuanced story. Both track nearly identically through 50k steps (lr=2e-4: -3, lr=6.3e-4: +8), then diverge: lr=2e-4 pulls ahead, reaching 64 at 75k and 132 at 100k, while the zoo baseline reaches 54 at 75k and 100 at 100k. Neither curve has flattened by 100k, meaning neither has converged within the training budget. The lower learning rate ultimately outperforms the zoo baseline on both mean return (150.4 vs. 105.0) and variance (std=86.0 vs. 139.9), consistent with the bias-variance framing. The zoo baseline fits more aggressively to recent experience and accumulates higher variance across seeds, while lr=2e-4 generalizes more reliably and, given more budget, would likely extend its lead further.

### 3.3 Exploration Fraction Sweep

The exploration fraction controls the proportion of total timesteps over which epsilon decays from 1.0 to its final value. Three values were tested: 0.06 (half of baseline), 0.12 (zoo baseline), and 0.24 (twice baseline). All other hyperparameters were held at zoo baseline values.

![Figure 3: Exploration fraction sweep](../img/sweep_exploration_learning_curves.png)

*Figure 3. Smoothed mean episode return with 95% confidence interval across 20 seeds for each exploration fraction condition. All other hyperparameters held at zoo baseline values.*

All three conditions track closely through 75,000 steps, reaching smoothed means of 66 (ef=0.06), 54 (ef=0.12), and 59 (ef=0.24). After 75k the curves diverge sharply: ef=0.06 and the baseline continue rising to 94 and 100 respectively, while ef=0.24 collapses to 13. The ef=0.06 condition is the slowest to get started, lagging behind both others through the first 25,000 steps, but recovers and finishes just below the baseline, suggesting that less exploration delays early learning without meaningfully changing the final outcome within this budget. The ef=0.24 collapse is concentrated entirely in the last quarter of training, after epsilon has fully decayed and the policy is operating near-greedily. The aggregate statistics (mean=15.4, std=322.7) make ef=0.24 look categorically bad, but Figure 3 shows the damage is confined to the final quarter of training, which is why those statistics are misleading.

### 3.4 Target Update Interval Sweep

The target update interval controls how frequently the target network weights are copied from the online network. Three values were tested: 125 (half of baseline), 250 (zoo baseline), and 500 (twice baseline). All other hyperparameters were held at zoo baseline values.

![Figure 4: Target update interval sweep](../img/sweep_tui_learning_curves.png)

*Figure 4. Smoothed mean episode return with 95% confidence interval across 20 seeds for each target update interval condition. All other hyperparameters held at zoo baseline values.*

All three conditions follow similar trajectories, but the differences in variance are meaningful. More frequent updates (tui=125) produce comparable mean return to the baseline (104.2 vs. 105.0) but higher variance (std=170.2 vs. 139.9). Less frequent updates (tui=500) degrade both mean return (44.2) and increase variance (std=193.2). The interpretation is consistent with the bias-variance framing: more frequent target updates mean the TD target shifts more often, causing the Q-network to fit more tightly to a moving signal rather than a stable one, increasing variance across seeds. Infrequent updates introduce a different problem: the target network becomes stale relative to the online network, slowing credit propagation and degrading mean return. The zoo baseline of tui=250 sits in a reasonable middle ground, though none of the three conditions reach the solved threshold reliably within the budget.

### 3.5 Ablation: No Experience Replay

This ablation tests the contribution of experience replay by replacing the 50,000-step replay buffer with a buffer of size one, forcing the network to train on each transition exactly once before discarding it. All other hyperparameters were held at zoo baseline values.

![Figure 5: No experience replay ablation](../img/ablation_no_replay_learning_curves.png)

*Figure 5. Smoothed mean episode return with 95% confidence interval across 20 seeds. No experience replay condition uses a buffer of size one; baseline uses a 50,000-step replay buffer.*

Without experience replay, the agent shows brief early improvement, reaching approximately -80 by 35,000 steps, before degrading back to -180 by 100,000 steps. The mean return of -177.3 (std=136.0) is comparable to the actor-critic baseline (-170.4), confirming that without replay, DQN's neural function approximator offers no meaningful advantage over a policy gradient approach. The degradation after 35,000 steps is particularly telling: the network initially benefits from the structure of the Q-learning update but cannot sustain learning once the correlated gradient signal causes weight updates to interfere with one another. Experience replay is not a performance enhancement on top of a working algorithm; it is a prerequisite for stable learning.

### 3.6 Ablation: No Target Network

This ablation tests the contribution of the target network by setting the target update interval to 100,000 steps, effectively freezing the target for the entire training run. All other hyperparameters were held at zoo baseline values.

![Figure 6: No target network ablation](../img/ablation_no_target_learning_curves.png)

*Figure 6. Smoothed mean episode return with 95% confidence interval across 20 seeds. No target network condition uses tui=100,000 (frozen for the full training run); baseline uses tui=250.*

Without the target network, the agent learns slowly but does not collapse. The mean return of -43.8 (std=37.1) is substantially below the baseline (105.0) but well above the no-replay condition (-177.3). The confidence interval is notably tight, reflecting low variance across seeds. This pattern confirms the asymmetry established in Section 1: the target network stabilizes learning but is not load-bearing in the way experience replay is. A frozen target network means TD targets never shift, which actually reduces one source of variance, but at the cost of slow credit propagation as the target falls increasingly out of date with the online network. The result is a stable but low-ceiling policy.

### 3.7 Summary

![Figure 7: Baseline DQN final return distribution](../img/baseline_distribution.png)

*Figure 7. Final return distribution across 20 seeds for the baseline DQN configuration (mean of last 10 episodes per seed). The rug plot shows individual seed outcomes. Mean (105) is pulled well below the median (160) by a sparse negative tail, giving a skewness of -1.60. The distribution is wide and left-skewed, not normally distributed around the mean.*

Figure 7 makes the single-run problem concrete. The majority of seeds land between 130 and 250, clustering near or above the solved threshold, but a sparse negative tail pulls the mean down to 105, well below the median of 160. A practitioner running a single seed could land anywhere in that range: a lucky seed near 250 would suggest reliable convergence, an unlucky one near -300 would suggest the algorithm fails entirely, and neither conclusion would be correct. The published zoo number of 136.79 is one draw from this distribution. The skewness of -1.60 quantifies what the plot shows visually: the distribution is not symmetric around the mean, and the mean is not a reliable summary of what the algorithm typically achieves. This is a general property of DQN, not specific to the baseline configuration: any single learning curve is a sample from a wide distribution, and evaluating on one seed conflates algorithm performance with random initialization.

| Configuration | Mean | Std | Min | Max |
|---|---|---|---|---|
| Baseline DQN (zoo config) | 105.0 | 139.9 | -291.7 | 247.6 |
| LR = 2e-4 | 150.4 | 86.0 | -33.7 | 255.1 |
| LR = 2e-3 | -335.4 | 542.2 | -2132.2 | 187.6 |
| Exploration fraction = 0.06 | 114.8 | 131.3 | -384.3 | 244.3 |
| Exploration fraction = 0.24 | 15.4 | 322.7 | -1263.5 | 228.5 |
| Target update interval = 125 | 104.2 | 170.2 | -476.4 | 212.6 |
| Target update interval = 500 | 44.2 | 193.2 | -714.9 | 234.7 |
| Ablation: no target network | -43.8 | 37.1 | -133.3 | 13.3 |
| Ablation: no experience replay | -177.3 | 136.0 | -628.2 | 20.3 |
| Actor-Critic W6 | -170.4 | 19.2 | -213.1 | -132.9 |

*Table 1. Final performance summary across all configurations (mean of last 10 episodes per seed, 20 seeds).*

The results across all experiments tell a consistent story. Experience replay is the prerequisite for stable learning: without it, DQN performs no better than actor-critic and degrades over time. The target network is complementary but secondary; removing it suppresses performance without causing collapse. Together, the ablations confirm that DQN's two innovations are not symmetric.

The sweeps extend this picture into the bias-variance framing. Across learning rate, exploration fraction, and target update interval, configurations that fit more aggressively to recent experience produced higher variance across seeds, while more conservative configurations generalized more reliably. The zoo baseline, despite being tuned for this environment, is not the best configuration under a 20-seed evaluation: lr=2e-4 outperforms it on both mean return and variance. The zoo config appears optimized for a single run rather than reliable convergence. Notably, neither the zoo baseline nor lr=2e-4 has stabilized within the 100k budget: both curves are still rising at the end of training, meaning the learning rate comparison reflects where the curves stood at 100k, not where they ultimately land. No configuration in this study shows clear signs of overfitting; the more common failure mode is instability or slow convergence, not degradation from over-training.

The deeper finding is methodological. Figure 7 shows what a single published number cannot: the distribution of outcomes is wide and left-skewed, with the mean pulled well below the median by a sparse but severe negative tail. A practitioner reading only the zoo's 136.79 would expect reliability. The full 20-seed distribution tells a different story. Making DQN work reliably requires understanding where its variance comes from, not just what its expected return is.

## Section 4: Reflection

The most surprising result was that the zoo baseline, tuned specifically for LunarLander-v3, is not the best configuration under a 20-seed evaluation. lr=2e-4 outperforms it on both mean return (150.4 vs. 105.0) and variance (std=86.0 vs. 139.9). This was not obvious going in: the zoo config is a published, community-vetted hyperparameter set with a known performance number. What the single published run conceals is that a more conservative learning rate, slower to fit and slower to diverge, generalizes more consistently across initializations. That finding reframed how I read benchmark numbers: they are samples from a distribution, not characterizations of it.

The exploration fraction sweep produced a second surprise. All three conditions tracked within a few points of each other through 75,000 steps. I had expected that too much exploration would cause learning to plateau, with the agent spending too long exploring and not enough time exploiting what it had learned. Instead, the sweep showed something more disruptive: ef=0.24 tracks the other conditions closely through 75k steps, then destabilizes entirely in the final 25,000 steps, once epsilon has fully decayed and the policy is operating near-greedily. The damage is not a ceiling but a collapse, and it is entirely a late-training phenomenon.

Both results connect directly to Sutton and Barto's deadly triad. DQN sits in all three legs (function approximation, bootstrapping, and off-policy training), and the ablations confirm the theoretical prediction. Without experience replay, correlated gradient updates cause the Q-network to oscillate rather than converge, and performance degrades to the actor-critic level. The target network mitigates the bootstrapping leg by freezing TD targets, but it does not escape the triad: all three legs remain, and the instability is suppressed rather than resolved. The ablation results bear this out asymmetrically: removing replay causes collapse, removing the target network causes a stable but low-ceiling policy. One innovation is structural, the other is stabilizing.

The clearest limitation of this study is the training budget. Both lr=2e-4 and the zoo baseline were still rising at 100k steps. Drawing conclusions about which configuration converges to a higher ceiling requires seeing convergence, and this study did not. A budget of 200k or more would let the curves reach convergence rather than cutting off mid-climb. A second limitation is the DQN-versus-actor-critic comparison itself: DQN was trained on LunarLander-v3 with discrete actions, while actor-critic used LunarLander-Continuous-v3. The environments are similar but not identical, and the action space difference is a confound. A cleaner comparison would hold the environment constant.

If I were running this study again, I would extend the budget and evaluate on held-out episodes after training rather than reading performance off the training curve. DQN is off-policy: the behavior policy during training is epsilon-greedy, while the target policy is greedy. The cliff walking lab demonstrated this directly: Q-learning's training return understates the quality of the learned policy because epsilon-greedy behavior keeps dragging episodes into suboptimal outcomes. Evaluating with epsilon=0 after training would likely give a cleaner read on what the agent actually learned, separate from the noise introduced by exploration. I would also run the actor-critic on the discrete environment to make the baseline comparison clean.

What this study changed most concretely is how I think about evaluation. Before running 20 seeds, a single learning curve feels like evidence. After seeing Figure 7, with most seeds clustering near the solved threshold but a sparse negative tail pulling the mean to 105, a single curve feels like anecdote. The methodology is part of the finding: characterizing the full performance distribution, not just reporting a mean, is what separates a reliable algorithm from a lucky run.

## References

Mnih, V., Kavukcuoglu, K., Silver, D., Graves, A., Antonoglou, I., Wierstra, D., & Riedmiller, M. A. (2013). Playing Atari with Deep Reinforcement Learning. *arXiv preprint arXiv:1312.5602*.

Raffin, A. (2020). RL Baselines3 Zoo. GitHub. https://github.com/DLR-RM/rl-baselines3-zoo

Sutton, R. S., & Barto, A. G. (2020). *Reinforcement Learning: An Introduction* (2nd ed.). MIT Press.